# 06 — Harmony Feature Extraction

Extract **harmony** features per song (`song_id` aligned with the Stage 1 manifest).

On Kaggle, raw MP3s are optional. Default path: **approximate from log-mel** already on disk (Phase 2 practical path). Attach an audio dataset under `/kaggle/input/mtg-jamendo-audio` to use real librosa audio features.


## Kaggle setup (every notebook)

1. **Settings → Internet → On** (right sidebar). Without this you get `Could not resolve host: github.com`.
2. **Add Data** (if this is not notebook `00` in the same session):
   - Attach the dataset you saved from notebook `00` (`mtg-instrument-cache`), **or**
   - Keep running inside the **same** Kaggle notebook after `00` (same `/kaggle/working`).
3. Each Kaggle notebook starts with an **empty** `/kaggle/working`. Files from a previous notebook are gone unless you attached them under `/kaggle/input`.
4. This bootstrap cell **auto-finds** files in `/kaggle/working` and `/kaggle/input`, copies a cache into working if needed, and **re-downloads annotations** if split TSVs are missing.


## Step 0 — Packages (`librosa` is slow on CPU; that is OK for this notebook)


In [ ]:
!pip install -q librosa soundfile tqdm


## Step 1 — Bootstrap paths


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, urllib.request
import numpy as np
import pandas as pd

WORKING_ROOT = Path("/kaggle/working/MTG_Instrument")
INPUT_BASE = Path("/kaggle/input")
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host: str = "github.com", port: int = 443, timeout: float = 5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    """MTG ids are 7-digit zero-padded (track_0000948 → 0000948)."""
    m = re.search(r"(\d+)", str(raw))
    if not m:
        return None
    return f"{int(m.group(1)):07d}"


def _find_file(name: str, bases: list[Path]) -> Path | None:
    for base in bases:
        if not base.exists():
            continue
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


def discover_input_root() -> Path | None:
    """Find a previous notebook-00 output or MTG data folder under /kaggle/input."""
    if not INPUT_BASE.exists():
        return None
    for marker in [
        "song_manifest.csv",
        "autotagging_genre-train.tsv",
        "autotagging_genre.tsv",
        ".shard_00_done",
    ]:
        hit = _find_file(marker, [INPUT_BASE])
        if hit is None:
            continue
        if marker == "song_manifest.csv":
            return hit.parents[1]  # .../MTG_Instrument/dataset/song_manifest.csv
        if marker == "autotagging_genre-train.tsv":
            # .../annotations/splits/split-0/file  OR  .../data/splits/split-0/file
            p = hit
            for _ in range(6):
                if (p / "dataset").exists() or p.name in {"MTG_Instrument", "data"}:
                    return p if p.name != "data" else p
                p = p.parent
            return hit.parents[2]
        if marker == "autotagging_genre.tsv":
            parent = hit.parent
            if parent.name == "annotations":
                return parent.parent
            return parent  # MTG data/
        if marker == ".shard_00_done":
            return hit.parents[2]  # .../MTG_Instrument/dataset/logmel_songs/.shard
    for p in INPUT_BASE.rglob("MTG_Instrument"):
        if p.is_dir():
            return p
    return None


def copy_cache_into_working(src: Path) -> None:
    """/kaggle/input is read-only — copy into working so later cells can write."""
    WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying cache {src} → {WORKING_ROOT} (may take a few minutes)...")
    for item in src.iterdir():
        dest = WORKING_ROOT / item.name
        if dest.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, dest)
        else:
            shutil.copy2(item, dest)
    print("Copy done.")


def ensure_annotations(ann_dir: Path) -> Path:
    """Make sure split-0 TSVs exist; wget them if this is a fresh Kaggle session."""
    train = ann_dir / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if train.exists():
        return ann_dir

    # maybe files are flat, or under /kaggle/input with a different layout
    hit = _find_file("autotagging_genre-train.tsv", [ann_dir, INPUT_BASE, Path("/kaggle/working")])
    if hit is not None:
        dest = ann_dir / "splits" / "split-0" / hit.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if hit.resolve() != dest.resolve():
            shutil.copy2(hit, dest)
        # copy sibling split files from the same folder
        for name in [
            "autotagging_genre-validation.tsv",
            "autotagging_genre-test.tsv",
            "autotagging_instrument-train.tsv",
            "autotagging_instrument-validation.tsv",
            "autotagging_instrument-test.tsv",
        ]:
            sib = hit.parent / name
            if sib.exists():
                shutil.copy2(sib, dest.parent / name)
        genre_full = _find_file("autotagging_genre.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if genre_full:
            shutil.copy2(genre_full, ann_dir / "autotagging_genre.tsv")
        inst_full = _find_file("autotagging_instrument.tsv", [hit.parents[2] if len(hit.parents) > 2 else hit.parent, INPUT_BASE])
        if inst_full:
            shutil.copy2(inst_full, ann_dir / "autotagging_instrument.tsv")
        print("Recovered split files from", hit.parent)
        return ann_dir

    if not check_internet():
        raise FileNotFoundError(
            "Split TSVs not found and Internet is OFF.\n"
            "Do ONE of:\n"
            "  A) Settings → Internet → On, re-run this cell (auto-download)\n"
            "  B) Add Data → attach notebook-00 output dataset (mtg-instrument-cache)\n"
            "  C) Stay in the SAME Kaggle session after running notebook 00"
        )

    print("Split TSVs missing — downloading official MTG annotations...")
    n = 0
    for rel in NEEDED_ANN:
        dest = ann_dir / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        url = f"{RAW_ANN}/{rel}"
        print("  wget", url)
        urllib.request.urlretrieve(url, dest)
        n += 1
    print(f"Downloaded {n} annotation files into {ann_dir}")
    return ann_dir


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    candidates = [
        ANN_DIR / "splits" / "split-0" / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / f"autotagging_{subset}-{split}.tsv",
        ANN_DIR / "splits" / "split-0" / f"{split}.tsv",
        ANN_DIR / f"{split}.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        found = _find_file(f"autotagging_{subset}-{split}.tsv", [ANN_DIR, INPUT_BASE, Path("/kaggle/working")])
        path = found
    if path is None:
        raise FileNotFoundError(
            f"No split file for {subset}/{split}.\n"
            "Re-run the bootstrap cell after enabling Internet, or attach notebook-00 output."
        )
    df = pd.read_csv(path, sep="\t")
    col = "TRACK_ID" if "TRACK_ID" in df.columns else df.columns[0]
    ids = set()
    for v in df[col].astype(str):
        tid = normalize_track_id(v)
        if tid:
            ids.add(tid)
    print(f"{split:12s}  {len(ids):6d} ids   ← {path}")
    return ids


ONLINE = check_internet()
print("Internet reachable:", ONLINE)

input_root = discover_input_root()
print("Discovered /kaggle/input cache:", input_root)

if input_root is not None and not (WORKING_ROOT / "annotations").exists() and not (WORKING_ROOT / "dataset" / "song_manifest.csv").exists():
    # If input looks like MTG_Instrument, copy it; if it looks like MTG data/, copy into annotations
    if (input_root / "dataset").exists() or (input_root / "annotations").exists():
        copy_cache_into_working(input_root)
    elif (input_root / "splits").exists() or (input_root / "autotagging_genre.tsv").exists():
        dest = WORKING_ROOT / "annotations"
        dest.mkdir(parents=True, exist_ok=True)
        for rel in NEEDED_ANN:
            s = input_root / rel
            if not s.exists():
                s = input_root / Path(rel).name
            if s.exists():
                d = dest / rel
                d.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(s, d)
                print("copied", d)

ROOT = WORKING_ROOT
ROOT.mkdir(parents=True, exist_ok=True)
MEL_DIR = ROOT / "dataset" / "logmel_songs"
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"

for p in [MEL_DIR, ANN_DIR, FEAT_DIR, CKPT_DIR, RESULTS_DIR, ROOT / "dataset"]:
    p.mkdir(parents=True, exist_ok=True)

ANN_DIR = ensure_annotations(ANN_DIR)

print("ROOT     =", ROOT)
print("MEL_DIR  =", MEL_DIR, "npy=", len(list(MEL_DIR.rglob('*.npy'))))
print("ANN_DIR  =", ANN_DIR)
print("split-0 train exists:", (ANN_DIR / "splits/split-0/autotagging_genre-train.tsv").exists())
print("MANIFEST =", MANIFEST, "exists=", MANIFEST.exists())


## Step 2 — Extract harmony and write `features/harmony/harmony_song.csv`


In [ ]:
import librosa
from tqdm.auto import tqdm

if not MANIFEST.exists():
    raise FileNotFoundError("Run notebook 01 first.")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
AUDIO_ROOT = Path("/kaggle/input/mtg-jamendo-audio")
WINDOW_SEC = 15.0
SR = 22050


def extract_from_audio(path: Path) -> dict:
    y, sr = librosa.load(path, sr=SR, mono=True, duration=60)
    hop = int(WINDOW_SEC * sr)
    vals = []
    for start in range(0, max(len(y) - hop, 0) + 1, hop):
        yw = y[start:start + hop]
        if len(yw) < hop // 2:
            break
        chroma = librosa.feature.chroma_stft(y=yw, sr=sr)
        tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(yw), sr=sr)
        row = {f"chroma_{i}_mean": float(np.mean(chroma[i])) for i in range(chroma.shape[0])}
        row.update({f"tonnetz_{i}_mean": float(np.mean(tonnetz[i])) for i in range(tonnetz.shape[0])})
        vals.append(row)
    return pd.DataFrame(vals).mean(numeric_only=True).to_dict() if vals else {}

def mel_proxy_features(S: np.ndarray) -> dict:
    bands = np.array_split(S, 12, axis=0)
    chroma = np.stack([b.mean() for b in bands])
    chroma = chroma / (chroma.sum() + 1e-6)
    row = {f"chroma_{i}_mean": float(chroma[i]) for i in range(12)}
    for i in range(6):
        row[f"tonnetz_{i}_mean"] = float(np.dot(chroma, np.cos(2 * np.pi * (i + 1) * np.arange(12) / 12)))
    return row


def features_from_mel(mel_path: Path) -> dict:
    S = np.load(mel_path)
    if S.ndim == 3:
        S = S.mean(0)
    return mel_proxy_features(S)

rows = []
for _, rec in tqdm(manifest.iterrows(), total=len(manifest)):
    sid = str(rec["song_id"])
    audio_candidate = None
    if "audio_path" in manifest.columns and pd.notna(rec.get("audio_path", None)):
        audio_candidate = Path(rec["audio_path"])
    elif AUDIO_ROOT.exists():
        hits = list(AUDIO_ROOT.rglob(f"*{sid}*.mp3"))
        audio_candidate = hits[0] if hits else None
    try:
        if audio_candidate and Path(audio_candidate).exists():
            feat = extract_from_audio(Path(audio_candidate))
            src = "audio"
        else:
            feat = features_from_mel(Path(rec["mel_abs"]))
            src = "mel_proxy"
    except Exception as e:
        print("fail", sid, e)
        continue
    feat.update({"song_id": sid, "source": src, "split": rec["split"]})
    rows.append(feat)

df = pd.DataFrame(rows)
out = FEAT_DIR / "harmony"
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "harmony_song.csv", index=False)
print(df.head())
print("wrote", out, "n=", len(df))
